# InferOps Week 2: Baseline GPU benchmark

**Goal:** measure the HF baseline engine on a real GPU and find its bottlenecks.
This is the "before" picture every later optimization is compared against.

**Setup:** Kaggle, **GPU T4 x2**, **Internet on**. Qwen2.5-3B-Instruct in FP16 on GPU 0.

**Budget:** about 45-60 GPU minutes for the full run. Stop the session (power button, top right) when done.

Run the cells top to bottom. Read the markdown notes between them: they explain what each number means.

## 1. Get the code

In [ ]:
import os
%cd /kaggle/working
if os.path.exists("inferops"):
    !git -C inferops pull
else:
    !git clone https://github.com/saideepa05/inferops.git
%cd /kaggle/working/inferops
!pip install -q -e ".[hf,dev]"
!python -m pytest -q 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Experiment configuration

Everything that defines the experiment lives here, so the report can record it.
`ignore_eos` is on in the load generator, so every request produces exactly `MAX_TOKENS` tokens.
That makes runs comparable.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
LABEL = "hf-fp16-t4"
DATASETS = ["short", "medium", "long"]   # ~20 / ~350 / ~2000 prompt tokens
CONCURRENCY = "1,4,8,16,32"
NUM_REQUESTS = 32                        # per concurrency level
MAX_TOKENS = 128
SLO_TTFT_MS, SLO_TPOT_MS = 1000, 100     # a request is "good" only if it meets both
OUT_DIR = "results/week2"
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Server helpers

In [ ]:
import subprocess, time, urllib.request

def start_server(cmd, env_extra, port, log_path, timeout_s=900):
    """Launch a server in the background and wait until /health answers."""
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, env=os.environ | env_extra, stdout=log, stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError(f"Server exited early. Last log lines:\n{open(log_path).read()[-3000:]}")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            print(f"Ready after {time.time() - start:.0f}s on port {port}")
            return proc
        except Exception:
            time.sleep(3)
    proc.terminate()
    raise TimeoutError(f"Server not ready after {timeout_s}s; see {log_path}")

def stop_server(proc):
    proc.terminate()
    proc.wait(timeout=60)

## 4. Start the baseline server on GPU 0

The first run downloads ~6 GB of weights. Then check memory with `nvidia-smi`.

**Predict before you look:** 3.09B parameters x 2 bytes (FP16) = ~6.2 GB of weights,
plus ~0.3-0.5 GB of CUDA context. GPU 1 should still be at 0.

In [ ]:
hf = start_server(
    ["python", "-m", "uvicorn", "inferops.server:app", "--port", "8000", "--log-level", "warning"],
    {"CUDA_VISIBLE_DEVICES": "0", "INFEROPS_BACKEND": "hf", "INFEROPS_MODEL_ID": MODEL_ID,
     "INFEROPS_DEVICE": "cuda", "INFEROPS_DTYPE": "float16"},
    port=8000, log_path="hf_server.log")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

## 5. Sanity check: one real request

In [ ]:
import httpx
t = time.perf_counter()
r = httpx.post("http://127.0.0.1:8000/v1/chat/completions", timeout=120, json={
    "messages": [{"role": "user", "content": "Explain the KV cache in two sentences."}],
    "max_tokens": 80, "temperature": 0}).json()
print(r["choices"][0]["message"]["content"])
print(r["usage"], f"| {time.perf_counter() - t:.2f}s")

## 6. Run the benchmark

Three prompt lengths x five concurrency levels. The serial engine makes this slow
(about 12-15 minutes per dataset), which is itself the finding.
Each run saves a JSON file with every request's timings plus GPU memory per level.

In [ ]:
for ds in DATASETS:
    print(f"\n===== dataset: {ds} =====")
    !python -m bench.loadgen --url http://127.0.0.1:8000 --dataset {ds} --concurrency {CONCURRENCY} --num-requests {NUM_REQUESTS} --max-tokens {MAX_TOKENS} --slo-ttft-ms {SLO_TTFT_MS} --slo-tpot-ms {SLO_TPOT_MS} --label {LABEL} --out {OUT_DIR}/{LABEL}_{ds}.json

## 7. Results table

In [ ]:
import json, pandas as pd

rows = []
for ds in DATASETS:
    report = json.load(open(f"{OUT_DIR}/{LABEL}_{ds}.json"))
    for lv in report["levels"]:
        rows.append({
            "dataset": ds, "concurrency": lv["concurrency"],
            "rps": lv["rps"], "out_tok_s": lv["output_tok_s"],
            "ttft_p50_ms": lv["ttft_s"]["p50"] * 1000, "ttft_p95_ms": lv["ttft_s"]["p95"] * 1000,
            "ttft_p99_ms": lv["ttft_s"]["p99"] * 1000,
            "tpot_p50_ms": lv["tpot_s"]["p50"] * 1000, "tpot_p95_ms": lv["tpot_s"]["p95"] * 1000,
            "goodput_rps": lv["goodput_rps"], "slo_pct": lv["slo_attainment"] * 100,
            "peak_mem_gb": lv["gpu"].get("0", {}).get("peak_mem_mib", float("nan")) / 1024,
            "errors": lv["errors"],
        })
df = pd.DataFrame(rows)
# Request rates are small numbers (0.18 req/s); one decimal would hide the differences.
df.round(1).assign(rps=df.rps.round(3), goodput_rps=df.goodput_rps.round(3))

## 8. Memory-bandwidth utilization (MBU)

Decode reads every weight once per token, so a bandwidth-bound engine has a hard ceiling:

`max tokens/s = memory bandwidth / (parameters x bytes per parameter)`

The T4's datasheet bandwidth is 320 GB/s. MBU compares what we achieved at
concurrency 1 (one request, pure decode speed) with that ceiling.

In [ ]:
from huggingface_hub import get_safetensors_metadata

n_params = sum(get_safetensors_metadata(MODEL_ID).parameter_count.values())
T4_BANDWIDTH = 320e9
BYTES_PER_PARAM = 2  # FP16

ceiling = T4_BANDWIDTH / (n_params * BYTES_PER_PARAM)
tpot_s = df.query("dataset == 'short' and concurrency == 1").tpot_p50_ms.iloc[0] / 1000
achieved = 1 / tpot_s
mbu = achieved * n_params * BYTES_PER_PARAM / T4_BANDWIDTH

print(f"Parameters:        {n_params / 1e9:.2f}B")
print(f"Ceiling:           {ceiling:.1f} tokens/s per sequence (bandwidth limit)")
print(f"Achieved:          {achieved:.1f} tokens/s (TPOT p50 {tpot_s * 1000:.1f} ms)")
print(f"MBU:               {mbu:.0%}")

**How to read MBU:** if it's well below ~70%, the GPU's memory system is mostly idle and the time
is going somewhere else, typically Python overhead and launching hundreds of small GPU kernels per token
(the HF loop runs 36 layers' worth of kernels one by one). That's overhead vLLM removes with CUDA graphs.
And at batch size 1, even a perfect engine can't exceed the ceiling. The only way past it is to
**decode many requests per weight read**: batching.

## 9. Charts

In [ ]:
import matplotlib.pyplot as plt

# Fixed categorical order: each dataset keeps its colour in every chart.
COLORS = {"short": "#2a78d6", "medium": "#eb6834", "long": "#1baf7a"}
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), facecolor=SURFACE)
panels = [
    ("ttft_p95_ms", "TTFT p95 (ms, log scale)", True),
    ("out_tok_s", "Output throughput (tokens/s)", False),
    ("tpot_p50_ms", "TPOT p50 (ms)", False),
]
for ax, (col, title, log) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    for ds in DATASETS:
        d = df[df.dataset == ds]
        ax.plot(d.concurrency, d[col], color=COLORS[ds], lw=2, marker="o", ms=7,
                markeredgecolor=SURFACE, markeredgewidth=2, label=ds)
    if col == "ttft_p95_ms":
        ax.axhline(SLO_TTFT_MS, color=MUTED, lw=1, ls="--")
        ax.annotate("TTFT SLO", (df.concurrency.max(), SLO_TTFT_MS), textcoords="offset points",
                    xytext=(-4, 5), ha="right", color=MUTED, fontsize=9)
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(df.concurrency.unique()), labels=sorted(df.concurrency.unique()))
    if log:
        ax.set_yscale("log")
    else:
        ax.set_ylim(bottom=0)
    ax.set_title(title, color=INK, loc="left", fontsize=11)
    ax.set_xlabel("Concurrent users", color=MUTED)
    ax.tick_params(colors=MUTED)
    ax.grid(True, color="#e6e5e0", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c9c8c2")
axes[0].legend(title="Prompt length", frameon=False)
fig.suptitle(f"HF baseline: {MODEL_ID}, FP16, Tesla T4", color=INK, x=0.01, ha="left", fontsize=13)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/baseline_charts.png", dpi=150, facecolor=SURFACE)
plt.show()

## 10. Write the report and package the results

In [ ]:
import platform
from datetime import datetime, timezone

def md_table(frame):
    cols = list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, row in frame.iterrows():
        cells = [(f"{v:.3f}" if c.endswith("rps") else f"{v:.1f}") if isinstance(v, float) else str(v)
                 for c, v in zip(cols, row)]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

report = f"""# Week 2 baseline report: {LABEL}

Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}

## Environment
| | |
|---|---|
| GPU | {torch.cuda.get_device_name(0)} (1 of {torch.cuda.device_count()} used) |
| Model | {MODEL_ID} ({n_params / 1e9:.2f}B params), FP16 |
| Engine | InferOps HF baseline (serial, max_concurrency=1) |
| torch / transformers | {torch.__version__} / {transformers.__version__} |
| Python | {platform.python_version()} |

## Workload
Closed-loop load, {NUM_REQUESTS} requests per level, {MAX_TOKENS} output tokens (ignore_eos),
temperature 0. SLO: TTFT <= {SLO_TTFT_MS} ms and TPOT <= {SLO_TPOT_MS} ms.

## Results
{md_table(df)}

![charts](baseline_charts.png)

## Bandwidth analysis
- Bandwidth ceiling at batch size 1: {ceiling:.1f} tokens/s
- Achieved: {achieved:.1f} tokens/s, so MBU = {mbu:.0%}

## Findings
_Write these yourself. See the questions at the end of the notebook._
"""
open(f"{OUT_DIR}/report.md", "w").write(report)
!cd /kaggle/working/inferops && zip -qr /kaggle/working/week2_results.zip {OUT_DIR}
print("Download week2_results.zip from the Output panel (right side, /kaggle/working)")

## 11. Stop the baseline server (frees GPU 0)

In [ ]:
stop_server(hf)

## 12. Optional preview: vLLM on GPU 1

Skip this if you're short on GPU hours. It installs vLLM (~5 minutes) and benchmarks the
**same model with the same tool** on the short dataset.

Notes:
- Installing vLLM can replace this environment's PyTorch, which is why it runs last.
- The T4 is an older architecture (compute capability 7.5). If vLLM refuses to start,
  the error in the log is a useful finding: it tells us which engine to use in Week 4.
- `gpu-memory-utilization 0.90` makes vLLM reserve 90% of VRAM up front for its KV cache.
  Compare its peak memory with the baseline's and explain the difference.

In [ ]:
!pip install -q vllm
!pip show vllm | head -2

In [ ]:
vllm = start_server(
    ["vllm", "serve", MODEL_ID, "--dtype", "half", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--port", "8001"],
    {"CUDA_VISIBLE_DEVICES": "1"}, port=8001, log_path="vllm_server.log", timeout_s=1200)
!python -m bench.loadgen --url http://127.0.0.1:8001 --dataset short --concurrency {CONCURRENCY} --num-requests {NUM_REQUESTS} --max-tokens {MAX_TOKENS} --slo-ttft-ms {SLO_TTFT_MS} --slo-tpot-ms {SLO_TPOT_MS} --label vllm-fp16-t4 --out {OUT_DIR}/vllm-fp16-t4_short.json
stop_server(vllm)
!cd /kaggle/working/inferops && zip -qr /kaggle/working/week2_results.zip {OUT_DIR}

In [ ]:
# If vLLM failed to start, the reason is in here:
!tail -40 vllm_server.log

## 13. Your bottleneck investigation

Answer these in the report's **Findings** section. This write-up is the Week 2 deliverable.

1. **Throughput vs concurrency.** Did output tokens/s change between 1 and 32 users? Why not?
2. **TTFT growth.** Predict TTFT p50 at 16 users from the single-user request time,
   then compare with the measurement. What does the match (or mismatch) tell you?
3. **Prompt length.** How much did prefill (TTFT at concurrency 1) grow from short to long?
   Did TPOT change with prompt length? Why? (Hint: what does each decode step read?)
4. **MBU.** What percentage of the T4's bandwidth does the baseline use? Where does the rest of the time go?
5. **Memory.** How much of the 15 GB did the baseline use? What could the unused memory hold?
6. **Goodput.** At what concurrency does the service stop meeting the SLO for most users?
   How many T4s would you need to serve 32 concurrent users within the SLO using this engine?
7. *(If you ran the vLLM preview)* How do vLLM's throughput and TTFT at 32 users compare?
   Which Week 1 limitation explains the difference?